# Superposition Geometry

This notebook explores the **geometric structure** of superposition:
how a neural network arranges feature representations when it has more
features than hidden dimensions.

We train a ToyModel with 50 features compressed into 15 hidden
dimensions, then examine four complementary views of the learned
geometry:

1. **W^T W (Gram matrix)** -- the dot-product structure between features
2. **Feature dimensionalities** -- how many hidden dims each feature uses
3. **Cosine similarity matrix** -- directional alignment between features
4. **Superposition metric** -- a single-number summary (rho_mm)

Finally, we sweep over feature density (p_active) to show that
**higher density drives down superposition**: when features frequently
co-activate, the network cannot afford to overlap them.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import torch

from occhio import ToyModel
from occhio.autoencoders import TiedLinearRelu
from occhio.distributions import SparseUniform
from occhio.model_grid import Axis, ModelGrid
from occhio.visualization import (
    FeatureDimensionalityByIndexPlot,
    SuperpositionIndicatorPlot,
    plot_representation,
)

pio.renderers.default = "png"
torch.manual_seed(42)

In [ ]:
N_FEATURES = 50
N_HIDDEN = 15
N_EPOCHS = 25000
LEARNING_RATE = 3e-4
P_ACTIVE = 0.01
BATCH_SIZE = 1024
DEVICE = "cpu"

# Decaying importances: feature 0 is most important, feature 49 least.
# This gives the model a clear priority ordering, making the geometry
# richer -- high-importance features get dedicated dimensions while
# low-importance features get superposed or dropped.
IMPORTANCE_DECAY = 0.97
IMPORTANCES = IMPORTANCE_DECAY ** torch.arange(N_FEATURES)

## 1. Train a ToyModel

We compress 50 sparse features into 15 hidden dimensions -- a 3.3x
bottleneck. With `p_active=0.01`, each feature fires in only 1% of
samples. This extreme sparsity gives the network room to superpose.

Feature importances decay geometrically (0.97^i), so the model
prioritizes early features and must decide which later features to
superpose versus drop.

In [ ]:
gen = torch.Generator(DEVICE).manual_seed(42)

dist = SparseUniform(N_FEATURES, p_active=P_ACTIVE, generator=gen, device=DEVICE)
ae = TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=gen, device=DEVICE)
model = ToyModel(dist, ae, device=DEVICE, importances=IMPORTANCES)

losses, _ = model.fit(
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

print(f"Final loss: {losses[-1]:.6f}")
print(f"Superposition (rho_mm): {model.superposition.item():.4f}")
print(
    f"Features alive (norm > 0.1): {(model.feature_norms > 0.1).sum().item()}/{N_FEATURES}"
)
print(
    f"Embedded features / hidden dim: {model.embedded_features_per_hidden_dimensions:.2f}"
)

## 2. W^T W (Gram matrix)

The Gram matrix `W^T W` shows the dot product between every pair of
feature embeddings. Diagonal entries are squared norms (how strongly
each feature is represented); off-diagonal entries reveal interference
between feature pairs.

`plot_representation` from `occhio.visualization` renders this as a
heatmap with a diverging colorscale: red = positive correlation,
blue = negative, white = orthogonal.

In [ ]:
fig = plot_representation(model, height=500, width=550)
fig.update_layout(title="W<sup>T</sup>W Gram Matrix (50 features, 15 hidden dims)")
fig.show()

## 3. Feature dimensionalities

Each feature's *dimensionality* measures how many hidden dimensions it
effectively occupies. A value of 1.0 means the feature has its own
dedicated direction; 0.5 means it shares a direction with one other
feature (an antipodal pair); lower values indicate denser packing.

With decaying importances, we expect a gradient: high-importance
features (low index) should claim more dimensions than low-importance
ones.

In [ ]:
norms = model.feature_norms.cpu().numpy()
dims = model.feature_dimensionalities.cpu().numpy()

alive_mask = norms > 0.1
n_alive = alive_mask.sum()

print(f"Alive features: {n_alive}/{N_FEATURES}")
print(f"Mean dimensionality (alive): {dims[alive_mask].mean():.4f}")
print(f"Total dimensionalities: {dims.sum():.2f} (hidden dims = {N_HIDDEN})")
print()

# Native bar chart of per-feature dimensionality
fig = FeatureDimensionalityByIndexPlot()(model, height=400, width=700)

# Reference lines for known geometric arrangements
for y_val, label in [(1.0, "Dedicated dim"), (0.5, "Digon"), (1 / 3, "Triangle")]:
    fig.add_hline(
        y=y_val,
        line_dash="dot",
        line_color="gray",
        annotation_text=label,
        annotation_position="top right",
        annotation_font_size=10,
    )

fig.update_layout(
    title="Feature Dimensionality by Index",
    yaxis_rangemode="tozero",
    showlegend=False,
)
fig.show()

## 4. Cosine similarity matrix

The cosine similarity matrix shows the directional alignment between
all pairs of feature embeddings. Unlike W^T W, this normalizes out
the norms so we see pure angular relationships.

Off-diagonal values near +/-1 indicate features sharing (or opposing)
the same direction -- the hallmark of superposition.

In [ ]:
cos_sim = model.cosine_similarity_matrix.cpu().numpy()

fig = go.Figure()
fig.add_trace(
    go.Heatmap(
        z=cos_sim,
        colorscale="RdBu_r",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar=dict(title="cos sim"),
    )
)
fig.update_layout(
    title="Cosine Similarity Between Feature Embeddings",
    xaxis_title="Feature j",
    yaxis_title="Feature i",
    yaxis_autorange="reversed",
    height=500,
    width=600,
)
fig.show()

## 5. Superposition metric (rho_mm)

The superposition metric is the mean of each feature's maximum
absolute cosine similarity to any other feature:

    rho_mm = (1/N) * sum_i max_{j != i} |cos(w_i, w_j)|

- rho_mm = 0: all features are orthogonal (no superposition)
- rho_mm = 1: every feature is perfectly aligned with some other feature

We also show the distribution of per-feature max cosine similarities.

In [ ]:
# Native gauge for the superposition metric
fig = SuperpositionIndicatorPlot()(model, height=200, width=300)
fig.show()

# Per-feature max cosine similarity (no native equivalent)
cos_sim = model.cosine_similarity_matrix.cpu().numpy()
cos_abs = np.abs(cos_sim)
np.fill_diagonal(cos_abs, 0.0)
max_cos_per_feature = cos_abs.max(axis=1)

rho_mm = model.superposition.item()
print(f"Superposition (rho_mm): {rho_mm:.4f}")
print(f"Min per-feature max cosine sim: {max_cos_per_feature.min():.4f}")
print(f"Max per-feature max cosine sim: {max_cos_per_feature.max():.4f}")

fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=max_cos_per_feature,
        nbinsx=20,
        marker_color="steelblue",
    )
)
fig.add_vline(
    x=rho_mm,
    line_dash="dash",
    line_color="red",
    annotation_text=f"rho_mm = {rho_mm:.3f}",
    annotation_position="top right",
)
fig.update_layout(
    title="Distribution of Per-Feature Max |Cosine Similarity|",
    xaxis_title="max |cos sim| to nearest neighbor",
    yaxis_title="Count",
    height=350,
    width=600,
)
fig.show()

## 6. Density sweep: how sparsity affects superposition

The central prediction of the toy models framework: **sparser features
enable more superposition**. When features rarely co-activate, the
network can overlap their representations without paying a large
reconstruction cost.

We sweep `p_active` from 0.001 (very sparse) to 0.1 (moderately
dense) using `ModelGrid` for efficient vectorized training.

With decaying importances, the effect is most visible in the
**alive feature count** and **embedded features per hidden dimension**:
the model drops low-importance features as density increases because
the interference cost of superposing frequently co-active features
outweighs their importance.

In [ ]:
SWEEP_DENSITIES = np.logspace(-3, -1, 8).tolist()


def create_model(params):
    gen = torch.Generator(DEVICE).manual_seed(42)
    return ToyModel(
        distribution=SparseUniform(
            N_FEATURES,
            p_active=params["Density"],
            generator=gen,
            device=DEVICE,
        ),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, generator=gen, device=DEVICE),
        importances=IMPORTANCES,
        device=DEVICE,
    )


grid = ModelGrid(
    create_model,
    axes=[Axis("Density", SWEEP_DENSITIES)],
)

grid.fit(
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    verbose=True,
)

print("\nSweep complete.")

## 7. Superposition vs density

We extract the superposition metric and related geometry measurements
from each trained model and plot them against density.

In [ ]:
densities = SWEEP_DENSITIES
rho_values = []
alive_values = []
mean_dim_values = []
embedded_per_hidden = []

print(f"{'p_active':>10} {'rho_mm':>8} {'alive':>6} {'mean_dim':>10} {'feat/hid':>10}")
print("-" * 50)

for i, m in enumerate(grid.models.flat):
    rho = m.superposition.item()
    alive = (m.feature_norms > 0.1).sum().item()
    mean_d = m.feature_dimensionalities.mean().item()
    eph = m.embedded_features_per_hidden_dimensions

    rho_values.append(rho)
    alive_values.append(alive)
    mean_dim_values.append(mean_d)
    embedded_per_hidden.append(float(eph))

    print(
        f"{densities[i]:>10.4f} {rho:>8.4f} {alive:>6.0f} {mean_d:>10.4f} {float(eph):>10.2f}"
    )

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Superposition (rho_mm)",
        "Alive Features (norm > 0.1)",
        "Mean Feature Dimensionality",
        "Embedded Features / Hidden Dim",
    ),
    horizontal_spacing=0.12,
    vertical_spacing=0.15,
)

fig.add_trace(
    go.Scatter(
        x=densities,
        y=rho_values,
        mode="lines+markers",
        marker=dict(size=8),
        line=dict(width=2, color="#2166ac"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=densities,
        y=alive_values,
        mode="lines+markers",
        marker=dict(size=8),
        line=dict(width=2, color="#4daf4a"),
    ),
    row=1,
    col=2,
)
fig.add_hline(
    y=N_FEATURES,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"all {N_FEATURES}",
    annotation_position="bottom right",
    annotation_font_size=10,
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=densities,
        y=mean_dim_values,
        mode="lines+markers",
        marker=dict(size=8),
        line=dict(width=2, color="#ff7f00"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=densities,
        y=embedded_per_hidden,
        mode="lines+markers",
        marker=dict(size=8),
        line=dict(width=2, color="#984ea3"),
    ),
    row=2,
    col=2,
)
fig.add_hline(
    y=1.0,
    line_dash="dot",
    line_color="gray",
    annotation_text="no superposition",
    annotation_position="bottom right",
    annotation_font_size=10,
    row=2,
    col=2,
)

for row in [1, 2]:
    for col in [1, 2]:
        fig.update_xaxes(type="log", title_text="p_active", row=row, col=col)

fig.update_yaxes(title_text="rho_mm", range=[0, 1.05], row=1, col=1)
fig.update_yaxes(title_text="# alive", row=1, col=2)
fig.update_yaxes(title_text="dims / feature", row=2, col=1)
fig.update_yaxes(title_text="features / dim", row=2, col=2)

fig.update_layout(
    title_text=(f"Density Sweep: {N_FEATURES} features, {N_HIDDEN} hidden dims"),
    height=650,
    width=900,
    showlegend=False,
)
fig.show()

## Key takeaways

1. **Sparse features enable superposition.** At low density (p_active ~
   0.001), the model packs many features into 15 dimensions by
   extensively overlapping their representations. The
   embedded-features-per-hidden-dim ratio is well above 1.

2. **Higher density reduces superposition.** As p_active increases,
   features co-activate more often and interference becomes costly.
   The model responds by dropping low-importance features (reducing
   the alive count) and dedicating more dimensions per surviving
   feature.

3. **Feature dimensionality reveals structure.** Values near 0.5
   indicate antipodal pairs (digons); values near 1/3 indicate
   triangular arrangements; values near 1.0 mean dedicated dimensions.

4. **The embedded-features-per-hidden-dim ratio** tracks how efficiently
   the network uses its capacity. Values above 1.0 indicate
   superposition; the ratio decreases with increasing density as the
   network shifts from superposition to faithful representation.